# 12. Topological Data Analysis: Persistent Homology and Singular Learning Theory LLC

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Edelsbrunner & Harer (2008 *Persistent homology - a survey*); Carlsson (2009 *Topology and data*); Bubenik (2015 *Statistical topological data analysis using persistence landscapes*); Watanabe (2001, 2009 *Algebraic Geometry and Statistical Learning Theory*); Lau et al. (2023 *Quantifying Generalization in Deep Learning using Singular Learning Theory*).
- **Purpose:** Quantify representation manifold topology using Vietoris-Rips persistent homology, Betti-0 connected component filtrations, persistence landscapes, and estimate the true singular complexity via the Singular Learning Theory Local Learning Coefficient (LLC hat{lambda}).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [11. Statistical Model Comparison Bayesian RoPE Friedman and Bootstrap](11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.ipynb) $\to$ `🔗 Downstream Syntheses:` [01. Adversarial Robustness FGSM PGD TRADES GCG and Smoothing](../11_ai_safety_privacy_and_verification/01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Never use classical regular model selection criteria (AIC, BIC, parameter count $d/2$) for deep networks because true neural parameter-function maps are non-identifiable and singular; avoid computing high-order Vietoris-Rips simplices on large unpruned point clouds ($N > 1000$ causes combinatorial memory explosion).

## Mathematical Foundations & Governing Equations

1. **Topological Feature Lifespan**: $\text{pers}_i = d_i - b_i$
2. **Level 1 Persistence Landscape**: $\lambda_1(t) = \max_{i=1}^M \max(0, \min(t - b_i, d_i - t))$
3. **Local Learning Coefficient (LLC)**: $\hat{\lambda}(w^*) = \frac{n \beta}{2} \mathbb{E}_{w \sim \text{SGLD}} [L_n(w) - L_n(w^*)]$
4. **Vietoris-Rips Filtration $\beta_0$**: $\beta_0(\epsilon) = |\text{Connected Components}(D \le \epsilon)|$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/12_topological_data_analysis_persistent_homology_and_singular_learning_llc.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_lifespans(pairs):
    return pairs[:, 1] - pairs[:, 0]

def calc_landscape1(pairs, eval_pts):
    t = eval_pts.unsqueeze(0)
    b = pairs[:, 0:1]
    d = pairs[:, 1:2]
    tri = torch.clamp(torch.minimum(t - b, d - t), min=0.0)
    return torch.max(tri, dim=0).values

def calc_llc(losses, loss_star, n_samples=500.0, beta=1.0):
    return (float(n_samples) * float(beta) / 2.0) * torch.mean(losses - loss_star)

def calc_betti0(dists, epsilons):
    n, e_cnt = dists.shape[0], epsilons.shape[0]
    counts = torch.zeros(e_cnt)
    for idx in range(e_cnt):
        adj = dists <= float(epsilons[idx].item())
        vis = [False] * n
        c = 0
        for i in range(n):
            if not vis[i]:
                c += 1
                q = [i]
                vis[i] = True
                while q:
                    curr = q.pop(0)
                    for nbr in range(n):
                        if bool(adj[curr, nbr].item()) and not vis[nbr]:
                            vis[nbr] = True
                            q.append(nbr)
        counts[idx] = float(c)
    return counts

In [ ]:
# Parity verification against golden reference tensors
lifespans = calc_lifespans(weights["birth_death_pairs"])
assert torch.max(torch.abs(lifespans - weights["lifespans_ref"])) < 1e-5, "Lifespans mismatch"

land = calc_landscape1(weights["birth_death_pairs"], weights["eval_points"])
assert torch.max(torch.abs(land - weights["landscape_ref"])) < 1e-5, "Landscape mismatch"

llc = calc_llc(weights["sample_losses"], weights["loss_star"], 500.0, 1.0)
assert torch.max(torch.abs(llc - weights["llc_ref"])) < 1e-5, "LLC mismatch"

betti0 = calc_betti0(weights["pairwise_distances"], weights["epsilon_thresholds"])
assert torch.max(torch.abs(betti0 - weights["betti0_ref"])) < 1e-5, "Betti-0 mismatch"
print("All TDA Persistent Homology and Singular Learning LLC assertions passed!")

In [ ]:
# Benchmarking TDA and Singular Learning operators
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_lifespans(weights["birth_death_pairs"])
    _ = calc_landscape1(weights["birth_death_pairs"], weights["eval_points"])
    _ = calc_llc(weights["sample_losses"], weights["loss_star"], 500.0, 1.0)
    _ = calc_betti0(weights["pairwise_distances"], weights["epsilon_thresholds"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused TDA + Landscape + LLC latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Regular Dimension d/2 Fallacy vs Singular Learning LLC hat{lambda}
# Model with d=1000 parameters has degenerate flat directions
d_params = 1000
classical_regular_dim = d_params / 2.0  # AIC/BIC assumes lambda = d/2 = 500

empirical_llc = calc_llc(weights["sample_losses"], weights["loss_star"], n_samples=500.0, beta=1.0).item()
print(f"Classical regular dimensionality d/2: {classical_regular_dim:.1f}")
print(f"Singular Learning Theory LLC hat{{lambda}}: {empirical_llc:.3f}")
assert empirical_llc < 0.1 * classical_regular_dim, "Deep models have huge parameter redundancy where hat{lambda} << d/2!"
assert empirical_llc > 0.0, "LLC must remain strictly positive!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Vietoris-Rips Persistent Homology** extracts topological invariants (connected components, loops) that are invariant to continuous coordinate deformations.
2. **Persistence Landscapes** convert multiscale topological persistence diagrams into functional vectors suitable for ML pipelines.
3. **Singular Learning Theory** resolves the overparameterization puzzle: true generalization complexity is governed by the Local Learning Coefficient $\hat{\lambda} \ll d/2$.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [12_topological_data_analysis_persistent_homology_and_singular_learning_llc.md](../../markdown/10_xai_and_ai_analysis_methods/12_topological_data_analysis_persistent_homology_and_singular_learning_llc.md)
- **PyTorch 2.x Module:** [12_topological_data_analysis_persistent_homology_and_singular_learning_llc.py](../../pytorch2/10_xai_and_ai_analysis_methods/12_topological_data_analysis_persistent_homology_and_singular_learning_llc.py)
- **Functional JAX Module:** [12_topological_data_analysis_persistent_homology_and_singular_learning_llc.py](../../jax/10_xai_and_ai_analysis_methods/12_topological_data_analysis_persistent_homology_and_singular_learning_llc.py)
- **Native Rust Candle Track:** [12_topological_data_analysis_persistent_homology_and_singular_learning_llc.rs](../../candle/10_xai_and_ai_analysis_methods/12_topological_data_analysis_persistent_homology_and_singular_learning_llc.rs)
- **Hardware Kernel (CPU / WGSL):** [12_topological_data_analysis_persistent_homology_and_singular_learning_llc.rs](../../kernels/10_xai_and_ai_analysis_methods/12_topological_data_analysis_persistent_homology_and_singular_learning_llc.rs)
- **Golden SafeTensors:** [12_topological_data_analysis_persistent_homology_and_singular_learning_llc.safetensors](../../weights/10_xai_and_ai_analysis_methods/12_topological_data_analysis_persistent_homology_and_singular_learning_llc.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 11. Statistical Model Comparison Bayesian RoPE Friedman and Bootstrap](11_statistical_model_comparison_bayesian_rope_friedman_and_bootstrap.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 01. Adversarial Robustness FGSM PGD TRADES GCG and Smoothing](../11_ai_safety_privacy_and_verification/01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.ipynb) &rarr;